# Evaluate with various inputs

## Objective

This notebook walks through how to use jsonl and csv files as inputs for evaluation, as well as both query/response and conversation-based inputs within those files. 

Note: When this notebook refers to 'conversations', we are referring to the definition of conversations defined [here](https://learn.microsoft.com/en-us/python/api/azure-ai-evaluation/azure.ai.evaluation.conversation?view=azure-python#attributes). This is a simplified variant on the broader Chat Protocol standard that is defined [here](https://github.com/microsoft/ai-chat-protocol)

## Time

You should expect to spend about 10 minutes running this notebook.

## Setup


In [20]:
# Install the Evaluation SDK package
# %pip install azure-ai-evaluation

### Imports
Run this cell to import everything that is needed for this sample

In [29]:
from azure.ai.evaluation import RougeScoreEvaluator, RougeType, evaluate
from pathlib import Path
import json

## Evaluator definition

We use an offline evaluator from the Azure AI Evaluation SDK so the notebook runs without external API calls. The built-in `RougeScoreEvaluator` compares each `response` against a `ground_truth` field, so we map `ground_truth` to the response column for this demo.

In [30]:
# Offline evaluator (no API calls) comparing response to ground_truth.
my_evaluator = RougeScoreEvaluator(rouge_type=RougeType.ROUGE_L)

## Datasets

Direct inputs into evaluators as shown above are useful for sanity checks. But for larger datasets we typically input the evaluator and a dataset file into the `evaluate` method. For that, we will need some data files.

Included in this sample directory are 3 files:
- qr_data.jsonl contains query/response inputs in jsonl format.
- qr_data.csv contains query/response inputs in csv format.
- conversation_data.jsonl contains conversation inputs in jsonl format.

Conversations and other complex inputs are not supported via csv inputs, so there is no corresponding "conversation_data.csv" file. Each file contains the same three query/response pairs, but in the conversation dataset, the second and third pairs are wrapped into a single, 4-turn conversation.

Double check the contents of these files by changing the print statement below. You might need to alter the `path_to_data` value depending on where your notebook is running:

In [31]:
# Auto-detect the outputs folder when running from notebooks/ or repo root.
path_to_data = Path.cwd().parent / "outputs"

# Define data path variables.
qr_js_data = path_to_data / "generated_quality_eval_20260305_121225.jsonl"
#qr_csv_data = path_to_data / "qr_data.csv"
#conversation_js_data = path_to_data / "conversation_data.jsonl"

# Change variable referenced here to check different files
print(f"Using data file: {qr_js_data.resolve()}")
if not qr_js_data.exists():
    raise FileNotFoundError(f"Missing data file: {qr_js_data}")
lines = qr_js_data.read_text().splitlines()
print(f"Total lines: {len(lines)}")
print("Preview (first 3 lines):")
print("\n".join(lines[:3]))

Using data file: C:\repo\ai-foundry-craftkit\Model_Usecases\quality_evals\outputs\generated_quality_eval_20260305_121225.jsonl
Total lines: 12
Preview (first 3 lines):
{"query": "Option 2 please.", "response": "{\"is_rpa_flow_related\":true,\"route_to\":\"current\",\"requires_prompt\":false}", "ground_truth": "is_rpa_flow_related=True; route_to='current'; requires_prompt=False", "system_prompt": "You are a classification assistant for a multi-step rpa_flow application. Given the current step and user message, classify the user's intent into one of the predefined categories.\n\nCURRENT STEP: step_d\nCURRENT STEP PURPOSE: The user is choosing an entry from a displayed list. Valid responses include picking an item by name or number.\n\nCLASSIFICATION GUIDELINES\n1. If the input directly answers or addresses the current step, then is_rpa_flow_related=True, route_to='current'.\n2. If the input requests navigating to a previous step (e.g. 'go back', 'start over', 'change my answer'), then is

## Evaluation

Now that we have some datasets and an evaluator, and can pass both of them into evaluate. Starting with query/response jsonl inputs:

In [33]:
evaluator_config = {
    "rouge": {
        "column_mapping": {
            "response": "${data.response}",
            "ground_truth": "${data.ground_truth}"
        }
    }
}

js_qr_output = evaluate(
    data=qr_js_data,
    evaluators={"rouge": my_evaluator},
    evaluator_config=evaluator_config,
    _use_pf_client=False,  # Avoid using PF dependencies to further simplify the example
)

eval_row_results = [row["outputs.rouge.rouge_f1_score"] for row in js_qr_output["rows"]]
metrics = js_qr_output["metrics"]

print(f"query/response jsonl results (ROUGE-L F1): {eval_row_results}")
print("aggregated metrics:")
print(json.dumps(metrics, indent=2))
metrics

2026-03-05 12:19:21 -0800   49408 execution.bulk     INFO     Finished 12 / 12 lines.
2026-03-05 12:19:21 -0800   49408 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.


======= Run Summary =======

Run name: "rouge_20260305_201921_602414"
Run status: "Completed"
Start time: "2026-03-05 20:19:21.602414+00:00"
Duration: "0:00:01.002892"



Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Combined Run Summary (Per Evaluator) =======

{
    "rouge": {
        "status": "Completed",
        "duration": "0:00:01.002892",
        "completed_lines": 12,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}


query/response jsonl results (ROUGE-L F1): [1.0, 1.0, 1.0, 0.9600000000000001, 1.0, 0.8571428571428571, 0.8571428571428571, 0.8571428571428571, 0.4102564102564102, 0.8421052631578948, 0.8421052631578948, 1.0]
aggregated metrics:
{
  "rouge.rouge_precision": 0.8693181818181818,
  "rouge.rouge_recall": 0.9340659340659342,
  "rouge.rouge_f1_score": 0.8854912923333976,
  "rouge.binary_aggregate": 0.92
}


{'rouge.rouge_precision': 0.8693181818181818,
 'rouge.rouge_recall': 0.9340659340659342,
 'rouge.rouge_f1_score': 0.8854912923333976,
 'rouge.binary_aggregate': 0.92}

## Conclusion

This sample has shown various ways to input data using `evaluate`, and the difference between query/response and conversation-based inputs. As the SDK is improved, more of the built-in evaluators will continue to support a larger variety of input schemes. We encourage users to leverage which ever options suit their needs.